# Thực Hành Chuyên Sâu: So Sánh Các Thuật Toán Hồi Quy & Kỹ Thuật Mở Rộng

Notebook này tổng hợp toàn bộ mã nguồn thực hành từ cơ bản đến nâng cao cho bài toán **Hồi quy (Regression)** trong Machine Learning & Thống kê ứng dụng:

1. **So sánh toàn diện 11 thuật toán Hồi quy:** Linear, Ridge ($L_2$), Lasso ($L_1$), Elastic Net, Polynomial, Decision Tree, Random Forest, Gradient Boosting, SVR, KNN và Neural Network (MLP).
2. **Hồi quy kháng nhiễu (Robust Regression):** Xử lý dữ liệu chứa ngoại lai (Outliers) nghiêm trọng với *Huber*, *RANSAC* và *Theil-Sen*.
3. **Hồi quy phân vị (Quantile Regression):** Ước lượng khoảng bất định (Prediction Intervals: P10 – P90) thay vì chỉ dự đoán giá trị trung bình.
4. **Mô hình tuyến tính tổng quát (GLM):** *Poisson* và *Gamma Regression* cho dữ liệu đếm và dữ liệu lệch phải dương.
5. **"Bẫy ngoại suy" (Extrapolation Trap) của mô hình cây:** Thí nghiệm chứng minh điểm yếu của Random Forest/GBDT khi dự báo xu hướng vượt ngoài miền huấn luyện và cách khắc phục bằng *Detrending*.
6. **Chẩn đoán mô hình & Diễn giải đặc trưng:** Phân tích *Residual Plot*, *Q-Q Plot* và *Permutation Importance*.

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Các thuật toán hồi quy
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    HuberRegressor, RANSACRegressor, TheilSenRegressor,
    QuantileRegressor, PoissonRegressor, GammaRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
import statsmodels.api as sm

# Cấu hình biểu đồ & font hỗ trợ đầy đủ tiếng Việt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['Noto Sans', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
np.random.seed(42)
print("Khởi tạo môi trường và thư viện thành công!")

Khởi tạo môi trường và thư viện thành công!

---
## Phần 1: So sánh toàn diện 11 thuật toán Hồi quy

Để đánh giá công bằng giữa các nhóm thuật toán, chúng ta tạo một bộ dữ liệu tổng hợp ($N = 1,200$ mẫu, $10$ đặc trưng) kết hợp cả:
- **Quan hệ tuyến tính** ($x_0, x_1$)
- **Quan hệ phi tuyến & tương tác biến** ($\sin(x_2)$, $x_3^2$, $x_0 \cdot x_4$)
- **Biến nhiễu không mang thông tin** ($x_6, x_7, x_8, x_9$)

> **Lưu ý thực hành:** Luôn đóng gói bước chuẩn hóa `StandardScaler()` vào bên trong `Pipeline` để tránh rò rỉ dữ liệu (*Data Leakage*) từ tập kiểm thử sang tập huấn luyện.

In [ ]:

n_samples, n_features = 1200, 10
X = np.random.normal(0, 1, size=(n_samples, n_features))

X[:, 5] = X[:, 0] * 0.95 + np.random.normal(0, 0.15, size=n_samples)


y = (
    4.5 * X[:, 0]
    - 3.0 * X[:, 1]
    + 5.0 * np.sin(1.5 * X[:, 2])
    + 2.2 * (X[:, 3] ** 2)
    + 2.5 * (X[:, 0] * X[:, 4])
    + np.random.normal(0, 1.5, size=n_samples)
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


models = {
    "Linear Regression (OLS)": LinearRegression(),
    "Ridge (L2)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (L1)": make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    "Elastic Net": make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5)),
    "Polynomial (Deg 2 + Ridge)": make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), Ridge(alpha=10.0)),
    "Decision Tree": DecisionTreeRegressor(max_depth=7, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, random_state=42),
    "SVR (RBF Kernel)": make_pipeline(StandardScaler(), SVR(C=10.0, epsilon=0.2)),
    "KNN Regressor (k=7)": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=7, weights='distance')),
    "Neural Network (MLP)": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
}

results = []
n_test, p_test = X_test.shape

for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time_ms = (time.perf_counter() - t0) * 1000
    
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - p_test - 1)
    
    results.append({
        "Thuật toán": name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "Adjusted R2": round(adj_r2, 4),
        "Thời gian (ms)": round(train_time_ms, 2)
    })

df_results = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
print(df_results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_results, x="RMSE", y="Thuật toán", palette="viridis", ax=axes[0])
axes[0].set_title("So sánh RMSE (Càng thấp càng tốt)", fontweight="bold")
axes[0].set_xlabel("RMSE")
axes[0].set_ylabel("")

sns.barplot(data=df_results, x="R2", y="Thuật toán", palette="magma", ax=axes[1])
axes[1].set_title("So sánh Hệ số xác định R² (Càng cao càng tốt)", fontweight="bold")
axes[1].set_xlabel("R² Score")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 1.05)

plt.tight_layout()
plt.show()

                Thuật toán    MAE   RMSE     R2  Adjusted R2  Thời gian (ms)
      Neural Network (MLP) 1.7112 2.1497 0.9280       0.9255        15693.17
      HistGradientBoosting 1.9874 2.6011 0.8946       0.8910         3663.34
          SVR (RBF Kernel) 2.0980 2.7982 0.8781       0.8738           78.31
Polynomial (Deg 2 + Ridge) 2.4666 3.2310 0.8374       0.8318            5.35
             Random Forest 2.4653 3.3415 0.8261       0.8201          987.61
       KNN Regressor (k=7) 3.2858 4.2819 0.7145       0.7046            2.32
                Lasso (L1) 3.5672 4.9085 0.6248       0.6118            2.38
             Decision Tree 3.7922 4.9307 0.6214       0.6083           10.15
               Elastic Net 3.6024 4.9326 0.6211       0.6080            2.47
                Ridge (L2) 3.5757 4.9457 0.6191       0.6059            2.57
   Linear Regression (OLS) 3.5805 4.9524 0.6180       0.6048            1.53


### Nhận xét từ kết quả thực nghiệm Phần 1:
- **Linear, Ridge, Lasso, Elastic Net** chỉ đạt $R^2 \approx 0.62$ vì không nắm bắt được các thành phần phi tuyến ($\sin(x_2)$, $x_3^2$, $x_0 x_4$).
- **Polynomial Regression (Bậc 2)** nhảy vọt lên $R^2 \approx 0.84$ nhờ tạo ra các đặc trưng bình phương và tương tác cặp biến.
- **Neural Network (MLP)** ($R^2 \approx 0.93$), **HistGradientBoosting** ($R^2 \approx 0.90$) và **SVR (RBF)** ($R^2 \approx 0.88$) đạt độ chính xác cao nhất vì tự động học được cả hàm sin lẫn tương tác phi tuyến phức tạp.

---
## Phần 2: Hồi quy kháng nhiễu (Robust Regression)

Trong dữ liệu thực tế (giá bất động sản, cảm biến IoT, giao dịch tài chính), luôn tồn tại các điểm **ngoại lai (Outliers)** do lỗi nhập liệu hoặc sự kiện bất thường.
- **OLS (Linear Regression)** tối thiểu hóa tổng bình phương sai số ($\sum e_i^2$), khiến một điểm ngoại lai cách xa đường hồi quy 100 đơn vị tạo ra lực kéo lớn gấp $10,000$ lần điểm bình thường.
- **Huber Regressor:** Dùng sai số bình phương cho các điểm nhỏ ($|e_i| \le \epsilon$) và chuyển sang sai số tuyệt đối tuyến tính cho các điểm ngoại lai ($|e_i| > \epsilon$).
- **RANSAC (Random Sample Consensus):** Lặp lại việc chọn ngẫu nhiên tập con dữ liệu, khớp mô hình và loại bỏ các điểm nằm ngoài ngưỡng dung sai (*outliers*).
- **Theil-Sen Estimator:** Tính trung vị của tất cả các độ dốc nối từng cặp điểm dữ liệu (chịu được tới ~29.3% điểm ngoại lai).

In [ ]:

rng = np.random.RandomState(42)
X_rob = np.linspace(0, 10, 120).reshape(-1, 1)
y_true_line = 3.2 * X_rob.ravel() + 5.0
y_rob = y_true_line + rng.normal(0, 1.5, size=120)


outlier_idx = rng.choice(120, size=18, replace=False)
y_rob[outlier_idx] += rng.uniform(25, 45, size=18)

robust_models = {
    "OLS (Linear Regression)": LinearRegression(),
    "Huber Regressor": HuberRegressor(epsilon=1.35),
    "RANSAC Regressor": RANSACRegressor(random_state=42),
    "Theil-Sen Estimator": TheilSenRegressor(random_state=42)
}

plt.figure(figsize=(11, 5.5))
inlier_mask = np.ones(120, dtype=bool)
inlier_mask[outlier_idx] = False

plt.scatter(X_rob[inlier_mask], y_rob[inlier_mask], color="steelblue", alpha=0.75, label="Dữ liệu chuẩn (Inliers)")
plt.scatter(X_rob[outlier_idx], y_rob[outlier_idx], color="crimson", marker="x", s=70, linewidths=2, label="Ngoại lai (Outliers - 15%)")
plt.plot(X_rob, y_true_line, "k--", linewidth=2, label="Đường hồi quy thực tế (Ground Truth)")

colors = ["orange", "green", "purple", "brown"]
for (name, m), c in zip(robust_models.items(), colors):
    m.fit(X_rob, y_rob)
    y_p = m.predict(X_rob)
    mae_clean = mean_absolute_error(y_true_line, y_p)
    plt.plot(X_rob, y_p, color=c, linewidth=2, label=f"{name} (MAE thực = {mae_clean:.2f})")

plt.title("Ảnh hưởng của Outliers lên OLS so với các thuật toán Robust Regression", fontweight="bold")
plt.xlabel("Biến độc lập X")
plt.ylabel("Biến mục tiêu Y")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## Phần 3: Hồi quy phân vị (Quantile Regression) — Ước lượng khoảng dự đoán

Các mô hình hồi quy thông thường chỉ ước lượng giá trị kỳ vọng trung bình $\hat{y} = E[Y|X]$. Tuy nhiên, khi dữ liệu có **phương sai thay đổi (Heteroscedasticity)** — ví dụ mức chi tiêu biến động mạnh hơn ở nhóm thu nhập cao — chúng ta cần biết **biên độ rủi ro**.

**Quantile Regression** sử dụng hàm mất mát **Pinball Loss**:
Khi huấn luyện ở các mức phân vị $q \in \{0.10, 0.50, 0.90\}$, ta thu được đường trung vị (P50) cùng dải tin cậy 80% nằm giữa P10 và P90.

In [ ]:

X_q = np.linspace(0.5, 10, 400).reshape(-1, 1)
y_q = 2.5 * X_q.ravel() + 4 * np.sin(X_q.ravel()) + np.random.normal(0, noise_scale)

quantiles = [0.10, 0.50, 0.90]
preds_q = {}

for q in quantiles:
    gbr_q = GradientBoostingRegressor(loss="quantile", alpha=q, n_estimators=150, max_depth=3, random_state=42)
    gbr_q.fit(X_q, y_q)
    preds_q[q] = gbr_q.predict(X_q)

coverage = np.mean((y_q >= preds_q[0.10]) & (y_q <= preds_q[0.90])) * 100

plt.figure(figsize=(11, 5.5))
plt.scatter(X_q, y_q, color="gray", alpha=0.35, s=20, label="Dữ liệu quan sát")
plt.plot(X_q, preds_q[0.50], color="navy", linewidth=2.2, label="Dự đoán Trung vị (P50)")
plt.plot(X_q, preds_q[0.90], color="darkorange", linestyle="--", linewidth=1.8, label="Phân vị trên (P90)")
plt.plot(X_q, preds_q[0.10], color="teal", linestyle="--", linewidth=1.8, label="Phân vị dưới (P10)")
plt.fill_between(X_q.ravel(), preds_q[0.10], preds_q[0.90], color="skyblue", alpha=0.25, label=f"Khoảng dự đoán 80% (Độ phủ thực tế: {coverage:.1f}%)")

plt.title("Quantile Regression: Dự đoán khoảng bất định khi phương sai thay đổi", fontweight="bold")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## Phần 4: Mô hình tuyến tính tổng quát (GLM — Poisson & Gamma Regression)

Giả định phân phối chuẩn của OLS bị vi phạm nghiêm trọng trong hai trường hợp phổ biến:
1. **Dữ liệu đếm (Count Data):** Số cuộc gọi tổng đài mỗi giờ, số đơn hàng mỗi ngày ($y \in \{0, 1, 2, \dots\}$). Dùng **Poisson Regression** với hàm liên kết $\log(\mu) = Xeta$ đảm bảo giá trị dự đoán luôn dương.
2. **Dữ liệu liên tục dương lệch phải (Positive Right-Skewed):** Chi phí bồi thường bảo hiểm, thời gian xử lý sự cố ($y > 0$). Dùng **Gamma Regression**.

In [ ]:

X_count = np.random.uniform(0, 5, size=(300, 1))

lambda_true = np.exp(0.5 + 0.65 * X_count.ravel())
y_count = np.random.poisson(lambda_true)

ols_count = LinearRegression().fit(X_count, y_count)
poisson_glm = PoissonRegressor(alpha=0.0).fit(X_count, y_count)

X_grid = np.linspace(0, 5.2, 200).reshape(-1, 1)

plt.figure(figsize=(11, 5))
plt.scatter(X_count, y_count, color="slategray", alpha=0.45, label="Số đơn hàng thực tế (Poisson)")
plt.plot(X_grid, np.exp(0.5 + 0.65 * X_grid.ravel()), "k--", linewidth=2, label="Kỳ vọng thực tế exp(0.5 + 0.65X)")
plt.plot(X_grid, ols_count.predict(X_grid), color="crimson", linewidth=2, label="OLS (Dự đoán sai dạng đường thẳng & có thể < 0)")
plt.plot(X_grid, poisson_glm.predict(X_grid), color="forestgreen", linewidth=2.5, label="Poisson GLM (Khớp chính xác cấu trúc hàm mũ)")

plt.title("So sánh OLS và Poisson Regression trên dữ liệu đếm", fontweight="bold")
plt.xlabel("Ngân sách Marketing (đơn vị)")
plt.ylabel("Số lượng đơn hàng")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Giá trị dự đoán nhỏ nhất của OLS tại X=0: {ols_count.predict([[0.0]])[0]:.2f} (Vô nghĩa với dữ liệu đếm!)")
print(f"Giá trị dự đoán nhỏ nhất của Poisson GLM tại X=0: {poisson_glm.predict([[0.0]])[0]:.2f}")

Giá trị dự đoán nhỏ nhất của OLS tại X=0: -3.65 (Vô nghĩa với dữ liệu đếm!)
Giá trị dự đoán nhỏ nhất của Poisson GLM tại X=0: 1.79

---
## Phần 5: "Bẫy ngoại suy" (Extrapolation Trap) của các mô hình cây & Giải pháp

Các thuật toán dựa trên cây (**Decision Tree, Random Forest, XGBoost, LightGBM**) chia không gian thành các vùng rời rạc và lấy trung bình giá trị trong từng lá. Do đó, chúng **không bao giờ dự đoán được giá trị cao hơn $\max(y_{	ext{train}})$ hoặc thấp hơn $\min(y_{	ext{train}})$**.

Nếu dữ liệu có xu hướng tăng trưởng theo thời gian ($y = a \cdot t + 	ext{mùa vụ}$), việc đưa thẳng vào Random Forest/GBDT để dự báo tương lai sẽ dẫn đến thảm họa "chạm trần".
- **Giải pháp chuẩn:** Dùng mô hình tuyến tính học xu hướng dài hạn (**Trend**), sau đó dùng mô hình cây học phần dư dao động phi tuyến (**Residuals**) — gọi là kỹ thuật **Detrending / Hybrid Modeling**.

In [ ]:

t = np.linspace(0, 15, 300).reshape(-1, 1)
y_ts = 2.0 * t.ravel() + 3.0 * np.sin(2.5 * t.ravel()) + np.random.normal(0, 0.5, size=300)
train_mask = t.ravel() <= 10.0
t_train, y_ts_train = t[train_mask], y_ts[train_mask]
t_test, y_ts_test = t[~train_mask], y_ts[~train_mask]

rf_pure = RandomForestRegressor(n_estimators=100, random_state=42).fit(t_train, y_ts_train)
gb_pure = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(t_train, y_ts_train)

trend_model = LinearRegression().fit(t_train, y_ts_train)
residuals_train = y_ts_train - trend_model.predict(t_train)
t_train_feat = np.column_stack([np.sin(2.5 * t_train), np.cos(2.5 * t_train)])
t_test_feat = np.column_stack([np.sin(2.5 * t_test), np.cos(2.5 * t_test)])
res_model = GradientBoostingRegressor(n_estimators=100, random_state=42).fit(t_train_feat, residuals_train)

y_hybrid_pred = trend_model.predict(t_test) + res_model.predict(t_test_feat)

plt.figure(figsize=(12, 5.5))
plt.scatter(t_train, y_ts_train, color="gray", alpha=0.5, s=18, label="Dữ liệu huấn luyện (t <= 10)")
plt.plot(t_test, y_ts_test, "k--", linewidth=2, label="Thực tế tương lai (t > 10)")
plt.plot(t_test, rf_pure.predict(t_test), color="crimson", linewidth=2.2, label="Random Forest thuần (Bị chạm trần!)")
plt.plot(t_test, gb_pure.predict(t_test), color="darkorange", linewidth=2.2, label="Gradient Boosting thuần (Bị chạm trần!)")
plt.plot(t_test, y_hybrid_pred, color="dodgerblue", linewidth=2.5, label="Hybrid (Linear Trend + GBDT Seasonality)")

plt.axvline(10.0, color="black", linestyle=":", linewidth=1.5)
plt.title("Bẫy ngoại suy (Extrapolation Trap) của Tree-based Models & Cách khắc phục", fontweight="bold")
plt.xlabel("Thời gian (t)")
plt.ylabel("Giá trị mục tiêu")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## Phần 6: Chẩn đoán giả định thống kê & Diễn giải mô hình

Sau khi huấn luyện mô hình tốt nhất (ví dụ **HistGradientBoosting** hoặc **MLP** ở Phần 1), kỹ sư dữ liệu không chỉ dừng lại ở điểm số $R^2$ mà cần kiểm tra:
1. **Residual Plot (Phần dư vs Giá trị dự đoán):** Các điểm sai số phải phân bố ngẫu nhiên quanh đường $0$ (không có hình phễu hay đường cong).
2. **Q-Q Plot (Quantile-Quantile):** Kiểm tra xem phần dư có tuân theo phân phối chuẩn hay không.
3. **Permutation Importance:** Đo mức độ sụt giảm của $R^2$ khi xáo trộn ngẫu nhiên từng cột đặc trưng trên tập kiểm thử (khách quan và đáng tin cậy hơn Gini Importance).

In [ ]:

best_model = models["HistGradientBoosting"]
y_test_pred = best_model.predict(X_test)
residuals = y_test - y_test_pred

perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=15, random_state=42, scoring="r2")
feature_names = [f"X{i}" + (" (nhiễu)" if i >= 6 else (" (cộng tuyến)" if i == 5 else "")) for i in range(n_features)]
sorted_idx = perm_imp.importances_mean.argsort()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].scatter(y_test_pred, residuals, alpha=0.6, color="teal", edgecolors="k", linewidth=0.4)
axes[0].axhline(0, color="crimson", linestyle="--", linewidth=1.8)
axes[0].set_title("1. Residual Plot (Kiểm tra phương sai)", fontweight="bold")
axes[0].set_xlabel("Giá trị dự đoán (Predicted)")
axes[0].set_ylabel("Phần dư (Residuals)")

stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("2. Normal Q-Q Plot (Phân phối chuẩn sai số)", fontweight="bold")

axes[2].barh(
    np.array(feature_names)[sorted_idx],
    perm_imp.importances_mean[sorted_idx],
    xerr=perm_imp.importances_std[sorted_idx],
    color="slateblue",
    alpha=0.85
)
axes[2].set_title("3. Permutation Importance (Tập Test)", fontweight="bold")
axes[2].set_xlabel("Mức giảm R² khi xáo trộn đặc trưng")

plt.tight_layout()
plt.show()

---
## Tổng kết & Lộ trình nghiên cứu tiếp theo

Qua các thực nghiệm trên, bạn có thể áp dụng quy trình chuẩn cho mọi bài toán hồi quy:
1. **Khám phá phân phối biến mục tiêu $Y$:** Nếu lệch phải/đếm $
ightarrow$ thử **GLM (Poisson/Gamma)** hoặc biến đổi $\log(1+y)$. Nếu nhiều outliers $
ightarrow$ dùng **Huber/RANSAC** hoặc mất mát `absolute_error` / `huber`.
2. **Thiết lập Baseline nhanh:** Bắt đầu với **Ridge Regression** (đã chuẩn hóa) để có mốc so sánh chuẩn.
3. **Nâng cấp mô hình:** Dùng **HistGradientBoosting / LightGBM / CatBoost** cho dữ liệu dạng bảng lớn, hoặc **MLP / SVR** cho dữ liệu trơn nhiều chiều.
4. **Kiểm tra ngoại suy & độ bất định:** Tách xu hướng (*Detrending*) nếu dự báo theo thời gian và dùng **Quantile Regression** nếu bài toán cần khoảng tin cậy.
5. **Mở rộng nâng cao nên tìm hiểu tiếp:**
   - Thư viện **SHAP (`shap.TreeExplainer`)** để giải thích đóng góp của từng đặc trưng cho từng dự đoán đơn lẻ.
   - **Generalized Additive Models (`pyGAM` / Splines)** để kết hợp tính diễn giải của hồi quy tuyến tính với khả năng uốn cong phi tuyến mượt mà.
   - **Bayesian Ridge Regression** để ước lượng tự động cả tham số chính quy hóa lẫn phương sai dự đoán.